# Archiving processed files from Source folder to Archive folder

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
try:
    dbutils.fs.ls ('/Volumes/mycata/stream/streaming/applyArchive/source/archive')
except Exception as e:
    if "FileNotFoundException" in str(e):
        dbutils.fs.mkdirs('/Volumes/mycata/stream/streaming/applyArchive/source/archive')
    else:
        raise e

In [0]:
try:
    dbutils.fs.ls ('/Volumes/mycata/stream/streaming/applyArchive/sink/checkpoint')
except Exception as e:
    if "FileNotFoundException" in str(e):
        dbutils.fs.mkdirs('/Volumes/mycata/stream/streaming/applyArchive/sink/checkpoint')
    else:
        raise e

In [0]:
schema=StructType([
    StructField('order_id', StringType()),
    StructField('timestamp', TimestampType()),
    StructField('customer',StructType([
        StructField('customer_id',IntegerType()),
        StructField('name', StringType()),
        StructField('email', StringType()),
        StructField('address', StructType([
            StructField('city', StringType()),
            StructField('country', StringType()),
            StructField('postal_code', StringType())
        ]))
    ])),
    StructField('items',ArrayType(
        StructType([
            StructField('item_id' , StringType()),
            StructField('price', DoubleType()),
            StructField('product_name',StringType()),
            StructField('quantity',IntegerType())
        ])
    )),
    StructField('metadata', ArrayType(
        StructType([
            StructField('key', StringType()),
            StructField('value', StringType())
        ])
    )),
    StructField('payment', StructType([
        StructField('method', StringType()),
        StructField('transaction_id', StringType())
    ]))
])

In [0]:
df=spark.readStream \
    .format('json') \
    .option('multiLine', True) \
    .schema(schema) \
    .option('cleanSource','archive') \
    .option('sourceArchiveDir', '/Volumes/mycata/stream/streaming/applyArchive/source/archive') \
    .load('/Volumes/mycata/stream/streaming/applyArchive/source/sourceData')

In [0]:
df=df.select(
          'order_id',
          'customer.customer_id',
          'customer.name',
          'customer.email',
          'customer.address.city',
          'customer.address.country',
          'customer.address.postal_code',
           explode_outer('items').alias('items'),
           explode_outer('metadata').alias('metadata'),
          'payment.method',
          'payment.transaction_id',
          'timestamp'
          )

In [0]:
df=df.select('order_id',
          'customer_id',
          'name',
          'email',
          'city',
          'country',
          'postal_code',
          'items.item_id',
          'items.price',
          'items.product_name',
          'items.quantity',
          'metadata.key',
          'metadata.value',
          'method',
          'transaction_id',
          'timestamp'
          )

In [0]:
wq= df.writeStream \
    .format('delta') \
    .outputMode('append') \
    .option('checkpointLocation', '/Volumes/mycata/stream/streaming/applyArchive/sink/checkpoint') \
    .option('path', '/Volumes/mycata/stream/streaming/applyArchive/sink/data') \
    .trigger(once=True) \
    .start()

In [0]:
print(wq.isActive)

True


In [0]:
%sql
SELECT * from delta.`/Volumes/mycata/stream/streaming/applyArchive/sink/data/`;

order_id,customer_id,name,email,city,country,postal_code,item_id,price,product_name,quantity,key,value,method,transaction_id,timestamp
ORD1001,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,I100,25.99,Wireless Mouse,2,campaign,back_to_school,Credit Card,TXN7890,2025-06-01T10:15:00.000Z
ORD1001,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,I100,25.99,Wireless Mouse,2,channel,email,Credit Card,TXN7890,2025-06-01T10:15:00.000Z
ORD1001,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,I101,15.49,USB-C Adapter,1,campaign,back_to_school,Credit Card,TXN7890,2025-06-01T10:15:00.000Z
ORD1001,501,John Doe,john@example.com,Toronto,Canada,M5H 2N2,I101,15.49,USB-C Adapter,1,channel,email,Credit Card,TXN7890,2025-06-01T10:15:00.000Z
ORD1002,502,Alice Smith,alice@example.com,Vancouver,Canada,V5K 0A1,I102,45.0,Bluetooth Keyboard,1,campaign,cyber_monday,PayPal,TXN7891,2025-06-01T10:30:00.000Z
ORD1002,502,Alice Smith,alice@example.com,Vancouver,Canada,V5K 0A1,I102,45.0,Bluetooth Keyboard,1,channel,affiliate,PayPal,TXN7891,2025-06-01T10:30:00.000Z
